[Open in Colab](https://colab.research.google.com/github/moonlovist/EEEA-ML/blob/main/sessions/session02/lab/en/02_Exercises_EN.ipynb)

# Lab 02 · Programming exercises
Ting Tan · EEEA

From regression to classification.

## Getting started
Run cells from top to bottom with **Shift+Enter**. No GPU or API key is needed. Data are included; no data upload is required.

**Colab:** open [Colab](https://colab.research.google.com/), choose **File → Upload notebook**, import this `.ipynb`, and connect to a Python runtime. Save/download your edited notebook before leaving.

**Local Jupyter:** in a terminal in the extracted folder:
```
python -m pip install -r requirements.txt
python -m notebook
```
If Jupyter is already configured, use your existing environment.

Run the import cell below. If later results look inconsistent, restart the kernel/runtime and run all cells in order. Read an error from its last line upward.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score
)
np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 11})

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    # Equivalent to 1 / (1 + exp(-z)), with stable evaluation.
    return np.exp(-np.logaddexp(0.0, -z))

def mean_bce(x, c, w, b):
    z = w * x + b
    return float(np.mean(np.logaddexp(0.0, z) - c * z))

print("Python / NumPy / Matplotlib / scikit-learn: OK")

Python / NumPy / Matplotlib / scikit-learn: OK


**Python reminders:** `np.array` stores a numerical vector; `*` multiplies element by element; `np.mean` computes a mean. `p >= 0.5` produces booleans; `np.sum(condition)` counts the `True` values. Indices start at 0.

The provided `sigmoid` and `mean_bce` functions implement the lecture formulas stably. All logarithms are natural.

## Your work
Complete **E1–E4** and their written questions. **B1–B3** extend the same concepts.

Edit your functions and answers, without changing the checks. `TODO` indicates unfinished code; `OK` indicates the supplied checks passed. This notebook is independent of the demonstrations.

Use NumPy for E1–E3; do not replace your functions with a library estimator or library classification metrics.

## E1 · A reusable linear update
Implement `linear_step(x, y, w, b, eta)`. Inputs x,y are nonempty 1D NumPy arrays of equal length. Return `(w_new, b_new)`.

Calculate residuals, then both **mean** gradients using old parameters, then update simultaneously.

Compute the second update from $(w,b)=(10.2,0.1)$ with $\eta=0.1$. Explain why changing w before calculating the gradient for b gives a different algorithm.

In [2]:
def linear_step(x, y, w, b, eta):
    # TODO: compute predictions, residuals and both mean gradients.
    # TODO: return the two updated parameters.
    return None

In [3]:
x = np.array([1., 2., 3.])
y = np.array([12., 19., 32.])
result = linear_step(x, y, 10.2, 0.1, 0.1)
if result is None:
    print("TODO E1")
else:
    assert np.allclose(result, [10.286666666666667, 0.15])
    other = linear_step(np.array([0., 2.]), np.array([1., 5.]), 0., 0., 0.1)
    assert np.allclose(other, [.5, .3]), "Check the mean and simultaneous update."
    assert np.allclose(linear_step(x, y, 10.2, .1, 0.), [10.2, .1])
    print("E1: OK | second update =", result)

TODO E1


**Answer:** _Write your explanation here._

## E2 · A logistic training function
Implement `fit_logistic(x, c, eta=0.1, updates=5000)` for matching nonempty 1D arrays, with labels 0/1.

- Initialise `w=b=0`; perform exactly `updates` batch updates.
- Return keys `w`, `b`, `losses`.
- Record initial loss and loss after every update: `len(losses) == updates + 1`.
- Use the supplied `sigmoid` and `mean_bce`.

Explain why `p-c` replaces the numerical residual in E1. What does a decreasing training curve tell you, and what does it not tell you?

In [4]:
def fit_logistic(x, c, eta=0.1, updates=5000):
    w, b = 0.0, 0.0
    losses = [mean_bce(x, c, w, b)]
    # TODO: repeat prediction, gradient calculation and simultaneous update.
    # TODO: append the loss AFTER every update.
    # TODO: return {"w": w, "b": b, "losses": losses}.
    return None

In [5]:
x_binary = np.arange(1., 7.)
c_binary = np.array([0, 0, 1, 0, 1, 1])
model = fit_logistic(x_binary, c_binary)
if model is None:
    print("TODO E2")
else:
    assert len(model["losses"]) == 5001
    assert np.isclose(model["losses"][0], np.log(2))
    assert np.allclose([model["w"], model["b"]], [1.2134457869, -4.2468229786])
    assert np.isclose(model["losses"][-1], .4129978434013483)
    assert np.all(np.diff(model["losses"]) <= 1e-12)
    one = fit_logistic(np.array([0., 1.]), np.array([1, 0]), updates=1)
    assert np.allclose([one["w"], one["b"]], [-.025, 0.])
    zero = fit_logistic(x_binary, c_binary, updates=0)
    assert len(zero["losses"]) == 1 and zero["w"] == zero["b"] == 0.
    print("E2: OK | final BCE =", model["losses"][-1])
    plt.plot(model["losses"], color="#e43d26")
    plt.xlabel("Update")
    plt.ylabel("Mean BCE")
    plt.tight_layout()
    plt.show()

TODO E2


**Answer:** _Write your explanation here._

## E3 · A classification report from boolean masks
Implement `classification_report_manual(y_true, probabilities, tau)` for matching nonempty 1D NumPy arrays and 0/1 labels.

Use $p\geq\tau$. Count TP, FP, FN, TN, then calculate accuracy, precision, recall, F1 and specificity. Return the exact keys listed in the code. Counts are integers. Use **0.0 if a metric's denominator is zero**.

Compare thresholds 0.5 and 0.8. Identify the groups forming the denominators of precision and recall. Why does equal accuracy not imply equivalent decisions?

In [6]:
def classification_report_manual(y_true, probabilities, tau):
    # TODO: predicted class is 1 when probability >= tau.
    # TODO: count TP, FP, FN, TN, then calculate the five metrics.
    # Return these exact keys:
    # TP, FP, FN, TN, accuracy, precision, recall, f1, specificity
    # Convention: return 0.0 for a metric whose denominator is zero.
    return None

In [7]:
truth = np.array([1, 1, 0, 1, 1, 0, 0, 0])
scores = np.array([.95, .85, .70, .60, .45, .30, .20, .10])
report = classification_report_manual(truth, scores, .5)
if report is None:
    print("TODO E3")
else:
    assert [report[k] for k in ["TP", "FP", "FN", "TN"]] == [3, 1, 1, 3]
    assert np.allclose([report[k] for k in
                       ["accuracy", "precision", "recall", "f1", "specificity"]],
                      [.75] * 5)
    high = classification_report_manual(truth, scores, .8)
    assert [high[k] for k in ["TP", "FP", "FN", "TN"]] == [2, 0, 2, 4]
    assert np.allclose([high["precision"], high["recall"], high["f1"]], [1, .5, 2/3])
    edge = classification_report_manual(np.array([1, 0]), np.array([.5, .5]), .5)
    assert edge["TP"] == edge["FP"] == 1, "Use >=, not >."
    empty_positive = classification_report_manual(np.array([0, 0]),
                                                 np.array([.1, .2]), .5)
    assert empty_positive["accuracy"] == 1 and empty_positive["precision"] == 0
    assert empty_positive["recall"] == empty_positive["f1"] == 0
    print("E3: OK")
    print("tau=0.5:", report)
    print("tau=0.8:", high)

TODO E3


**Answer:** _Write your explanation here._

## E4 · Train, select a threshold, then test
The supplied synthetic data have **200 training, 100 validation and 100 test cases**, each with a distance and binary label. They simulate the probability that a walk exceeds 30 minutes; they are not measurements of real people. Run the data cell.

Fit parameters on **training** using E2. Implement `choose_threshold` using E3: maximise **validation F1** among `[0.2, 0.35, 0.5, 0.65, 0.8]`. Break ties by choosing the **highest threshold**. Return one numerical threshold.

Never use test labels to fit parameters or select the threshold.

In [8]:
def make_walk_splits():
    rng = np.random.default_rng(2026)
    splits = {}
    for name, size in [("train", 200), ("validation", 100), ("test", 100)]:
        x = rng.uniform(0.5, 6.5, size)
        event_probability = sigmoid(1.2 * (x - 3.5))
        c = (rng.random(size) < event_probability).astype(int)
        splits[name] = (x, c)
    return splits

walks = make_walk_splits()
x_train, c_train = walks["train"]
x_val, c_val = walks["validation"]
print("Training:", len(x_train), "| Validation:", len(x_val))

Training: 200 | Validation: 100


In [9]:
def choose_threshold(y_val, p_val, candidates):
    # TODO: evaluate F1 on VALIDATION for every candidate.
    # TODO: return the candidate with highest F1.
    # If several candidates tie, choose the HIGHEST threshold.
    return None

In [10]:
candidates = np.array([.2, .35, .5, .65, .8])
frozen_model = None
trained = fit_logistic(x_train, c_train)
if trained is None or classification_report_manual(c_val, np.full(len(c_val), .5), .5) is None:
    print("TODO E2 / E3")
else:
    p_val = sigmoid(trained["w"] * x_val + trained["b"])
    tau = choose_threshold(c_val, p_val, candidates)
    if tau is None:
        print("TODO E4")
    else:
        assert tau in candidates
        validation_rows = []
        print("tau    validation F1")
        for candidate in candidates:
            value = classification_report_manual(c_val, p_val, candidate)["f1"]
            validation_rows.append((float(candidate), value))
            print(f"{candidate:.2f}   {value:.6f}")
        best_f1 = max(value for candidate, value in validation_rows)
        assert np.isclose(classification_report_manual(c_val, p_val, tau)["f1"], best_f1)
        assert tau == max(t for t, value in validation_rows if value == best_f1)
        assert choose_threshold(np.array([1, 0]), np.array([.9, .1]),
                                np.array([.2, .8])) == .8
        frozen_model = (float(trained["w"]), float(trained["b"]), float(tau))
        print("Frozen (w, b, tau) =", frozen_model)

TODO E2 / E3


### Final evaluation
After `(w, b, tau)` is frozen, run the test cell and report its results. Do not retune after seeing them.

Explain each split's role. Does maximising validation F1 guarantee maximal test F1?

In [11]:
if frozen_model is None:
    print("TODO E4: freeze the model using validation first.")
else:
    w_final, b_final, tau_final = frozen_model
    x_test, c_test = walks["test"]
    p_test = sigmoid(w_final * x_test + b_final)
    test_report = classification_report_manual(c_test, p_test, tau_final)
    print("TEST:", test_report)
    print("TEST ROC-AUC:", roc_auc_score(c_test, p_test))
    print("TEST AP:", average_precision_score(c_test, p_test))
    # Verify the manual report with an independent library.
    test_labels = (p_test >= tau_final).astype(int)
    assert np.isclose(test_report["f1"], f1_score(c_test, test_labels, zero_division=0))

TODO E4: freeze the model using validation first.


**Answer:** _Write your explanation here._

## B1 · Learning-rate experiment
Reuse E1. Start at `(10,0)` and run 30 updates for each rate `0.01, 0.1, 0.4`. Plot the three losses with a logarithmic vertical axis. Keep data and initial parameters identical. Explain slow progress and divergence.

In [12]:
# TODO B1: compare eta = 0.01, 0.1, 0.4 on the linear regression data.
# Use the same start (10, 0) and 30 updates. Plot the loss trajectories.

## B2 · Reconstruct ROC without `roc_curve`
Implement `roc_manual(y_true, scores)`; both classes are present. Start above all scores, then lower the threshold through **distinct scores in descending order**. Group ties. Return arrays `(fpr,tpr)` including `(0,0)` and `(1,1)`.

Calculate AUC by trapezoids and compare with 0.875. A tied positive–negative pair contributes half a win. Do not use trapezoidal PR integration to calculate AP.

In [13]:
def roc_manual(y_true, scores):
    # TODO: group equal scores at the same threshold.
    # Return arrays (fpr, tpr), starting at (0,0) and ending at (1,1).
    # Inputs contain both classes.
    return None

In [14]:
curve = roc_manual(truth, scores)
if curve is None:
    print("TODO B2")
else:
    fpr, tpr = curve
    # Trapezoids, written explicitly for compatibility with NumPy versions.
    auc_manual = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)
    assert np.isclose(auc_manual, .875)
    tied = roc_manual(np.array([0, 1]), np.array([.5, .5]))
    tfpr, ttpr = tied
    assert np.isclose(np.sum(np.diff(tfpr) * (ttpr[1:] + ttpr[:-1]) / 2), .5)
    print("B2: OK | AUC =", auc_manual)

TODO B2


## B3 · An apparently strong classifier
Construct 1,000 labels: 10 positive, 990 negative. Assign probability 0 to every case and use threshold 0.5. Reuse E3 to calculate accuracy, recall and F1. Is this model useful for detecting positives?

In [15]:
# TODO B3: 10 positives, 990 negatives; predict 0 for every case.
# Calculate accuracy, recall and F1, then interpret the result.

## Submission
Save as `Surname_Name_Lab02.ipynb` with completed E1–E4, outputs, written answers and extensions attempted. Restart the kernel/runtime and run all cells before submitting. Retain the selected threshold and final test report.

Passing checks does not replace an explanation of the algorithm and evaluation protocol.

## References
- [Stanford CS229: supervised learning](https://see.stanford.edu/materials/aimlcs229/cs229-notes1.pdf)
- [scikit-learn: confusion matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [scikit-learn: precision–recall](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html)